# Ocean surface freshwater flux by source — alternate spin-ups vs Full-CPL

The global ocean freshwater input split into its sources — rain, snow, evaporation, river and ice runoff, sea-ice
melt/growth, frazil (and land-ice / iceberg melt, zero in these runs) — and their sum, the net mass flux (panel (c) of
`2_ocn_sfc_compare_analysis`). During the FOSI segments of FC-FOSI-S1/S2 the atmosphere is a data atmosphere, so the
surface freshwater forcing differs from the coupled FC; this notebook shows by how much and through which terms.

Data: MPAS-Ocean global statistics in `post/time_series/ocn` (global integrals, kg s⁻¹, positive into the ocean),
converted to ocean-area means in mm yr⁻¹ with the global ocean area from MPAS-Analysis. "Net" is the sum of the sources:
the MPAS `netMassFlux` (used for panel (c) of notebook 2) is 0 in the first month after every restart, which biases
the annual mean of restart years by ~10 mm yr⁻¹ (every 5th year in the FC-FOSI runs).

Figures:
* `fw_flux_components_0001-0350.pdf` — annual fluxes per source (`SMOOTH`-yr running mean), FOSI segments shaded
* `fw_flux_differences.pdf` — FC-FOSI − FC mean per source over `PERIODS` (alternating phase, fully coupled phase)

Figures go to `FIG_DIR` (public_html), annual series to `DATA_DIR`.

In [ ]:
import os, sys
WF_ROOT = os.path.abspath("..")            # coupled_spinup_eval/
if WF_ROOT not in sys.path:
    sys.path.insert(0, WF_ROOT)
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
%matplotlib inline

import matplotlib.pyplot as plt

import numpy as np
import xarray as xr

from util.experiments import make_run_dict
from util.figures import publish
from util.fw_flux import COMPONENTS, ocean_area, read_fw_annual, plot_fw_components, plot_fw_differences

## Setup

In [ ]:
# ---- paths ----
MODEL_ROOT   = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings"     # case directories
OUTPUT_ROOT  = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3spinup_paper"   # published figures/ and data/
OUTPUT_URL   = "https://web.lcrc.anl.gov/public/e3sm/diagnostic_output/ac.szhang/v3spinup_paper"
FIG_DIR      = f"{OUTPUT_ROOT}/figures/fw_flux"
DATA_DIR     = f"{OUTPUT_ROOT}/data/fw_flux"               # cached/derived data (index: data/README.md)

MPAS_TS_FILE = "post/analysis/mpas_analysis/ts_0001-0350_climo_0301-0350/timeseries/mpasTimeSeriesOcean.nc"   # ocean area (under the FC case)

# ---- experiments: case directory names (under MODEL_ROOT) and model-year periods ----
EXPERIMENTS = {
    "Full-CPL": {
        "spin-up":   {"name": "20231209.v3.LR.piControl-spinup.chrysalis",      "period": "0001-2000"},
        "piControl": {"name": "v3.LR.piControl",                                "period": "0001-0500"},
    },
    "AltBG1": {
        "spin-up":   {"name": "20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0001-0220"},
        "piControl": {"name": "20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0221-0350"},
    },
    "AltBG2": {
        "spin-up":   {"name": "20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0001-0220"},
        "piControl": {"name": "20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0221-0350"},
    },
}
LABEL = {"Full-CPL": "FC", "AltBG1": "FC-FOSI-S1", "AltBG2": "FC-FOSI-S2", "AltBG3": "FC-FOSI-S3"}

# ---- analysis settings ----
EXPS        = ["Full-CPL", "AltBG1", "AltBG2"]   # runs shown; the first is the reference
COLORS      = ["black", "#cc0000", "#377eb8"]    # one per run
YEARS       = (1, 350)      # model years read from each spin-up case
SWITCH_YEAR = 221           # first fully coupled year of the FC-FOSI runs
FOSI_SEGMENTS = {           # FOSI (data-atmosphere) years of each FC-FOSI run (no atmosphere output)
    "AltBG1": [(21, 80), (91, 150), (161, 220)],
    "AltBG2": [(61, 220)],
}
PERIODS     = {"Years 1–220 (alternating)": (1, 220), "Years 221–350 (fully coupled)": (221, 350)}

In [ ]:
# Shared inputs/outputs
run_dict = make_run_dict(EXPERIMENTS, MODEL_ROOT)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
publish(OUTPUT_ROOT, FIG_DIR)
print("figures ->", FIG_DIR)
print("data    ->", DATA_DIR)
print("web     ->", FIG_DIR.replace(OUTPUT_ROOT, OUTPUT_URL))

## 1 Annual freshwater fluxes
Cached per run as `fw_flux_annual_<exp>_<years>.nc` in `DATA_DIR` (`FORCE = True` rebuilds).

In [ ]:
FORCE = False
AREA = ocean_area(os.path.join(MODEL_ROOT, run_dict[EXPS[0]]["spin-up"]["name"], MPAS_TS_FILE))
print(f"global ocean area: {AREA:.4e} m2")

fw = {}
for t in EXPS:
    cache = os.path.join(DATA_DIR, f"fw_flux_annual_{t}_{YEARS[0]:04d}-{YEARS[1]:04d}.nc")
    if os.path.isfile(cache) and not FORCE:
        fw[t] = xr.open_dataset(cache).load()
    else:
        fw[t] = read_fw_annual(os.path.join(MODEL_ROOT, run_dict[t]["spin-up"]["name"], "post/time_series/ocn"),
                               YEARS, AREA)
        fw[t].attrs["ocean_area_m2"] = AREA
        fw[t].to_netcdf(cache)
    print(f"{LABEL[t]:11s} mean net {float(fw[t]['net'].mean()):+7.2f} mm/yr "
          f"(MPAS netMassFlux {float(fw[t]['net_mpas'].mean()):+7.2f}, which is 0 in the first month after restarts)")
publish(DATA_DIR)

## 2 Figures
### Settings
`SHOW` = sources drawn (land-ice and iceberg melt are zero in these runs and left out).

In [ ]:
SHOW   = ["rain", "evap", "snow", "river", "irun", "seaice", "frazil", "net"]
SMOOTH = 10                 # running mean (yr) of the time series
TS_STYLE = dict(ncols=2, fontz=12.0, panel_letters=True)

### Plot

In [ ]:
show = {k: COMPONENTS.get(k, (k, "Net (sum of sources)")) for k in SHOW}
series = {LABEL[t]: dict(ds=fw[t], color=c) for t, c in zip(EXPS, COLORS)}
segments = {LABEL[t]: FOSI_SEGMENTS.get(t, []) for t in EXPS}
out = os.path.join(FIG_DIR, f"fw_flux_components_{YEARS[0]:04d}-{YEARS[1]:04d}.pdf")
fig, axes = plot_fw_components(series, show, smooth=SMOOTH, segments=segments, markers=[SWITCH_YEAR], **TS_STYLE, save=out)
plt.show()
print("web:", out.replace(OUTPUT_ROOT, OUTPUT_URL))

ref = fw[EXPS[0]]
diffs = {p: {LABEL[t]: (fw[t] - ref).sel(year=slice(*yrs)).mean("year") for t in EXPS[1:]} for p, yrs in PERIODS.items()}
for p, d in diffs.items():
    print(p); [print(f"  {r:11s}", "  ".join(f"{k}={float(v[k]):+6.1f}" for k in SHOW)) for r, v in d.items()]
out = os.path.join(FIG_DIR, "fw_flux_differences.pdf")
fig, axes = plot_fw_differences(diffs, show, colors={LABEL[t]: c for t, c in zip(EXPS, COLORS)}, save=out)
plt.show()

publish(FIG_DIR, DATA_DIR)
print("web:", out.replace(OUTPUT_ROOT, OUTPUT_URL))